# Семинар 5. Pandas: фильтрация, сортировка, группировка, агрегация, сводные таблицы

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-05.ipynb`, например `2MP9-Ivanov-Ivan-seminar-05.ipynb`.
- Ноутбук читает файл `datasets/legal/dtp-bryansk.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Описание данных — в `datasets/legal/README.md`.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-05-tasks.md` с формулировками и ноутбук `seminar-05-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 5».
- Шпаргалка по функциям: `reference/pandas.md`.

## 1. Вопросы, на которые маска не отвечает

На семинаре 4 таблица ДТП Брянской области отвечала на вопросы масками: сравнение столбца с числом или строкой, `sum` и `mean` по маске, `loc` с маской и списком столбцов. Сегодня та же таблица: 10 114 ДТП с пострадавшими за 2015–2025 годы, по одной строке на происшествие, номер ДТП в индексе.

In [ ]:
# Импортируем библиотеку pandas
import pandas as pd

In [ ]:
# Загрузка данных о ДТП в Брянской области из CSV-файла
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id')   # ДТП Брянской области
dtp.head(3)                                                              # первые три ДТП

Маска отвечает на вопрос об одном значении: сколько ДТП в Почепском районе и сколько из них с погибшими.

In [ ]:
pochep = dtp['district'] == 'Почепский район'                           # маска: Почепский район
pochep.sum(), (dtp.loc[pochep, 'dead_count'] > 0).sum()                  # ДТП и ДТП с погибшими

456 ДТП, из них 99 с погибшими. Районов в таблице 29, и на вопрос «в каком районе доля ДТП с погибшими выше» маска ответит только после 29 повторений. Вопросы, которые нужны для отчёта и с которыми масками не справиться:

1. Как менялось число ДТП и погибших по годам?
2. В каких районах доля ДТП с погибшими выше, чем в других?
3. Различаются ли виды ДТП по тяжести последствий?

Ответ на все три — группировка: таблица делится на группы по значению столбца, в каждой группе считается статистика, результаты собираются в новую таблицу. К группировке ведут три инструмента поменьше: отбор строк текстом, сортировка, новые столбцы. Порядок занятия такой же.

## 2. Фильтрация: `query` и строки

Маска из двух условий требует скобок вокруг каждого условия и знака `&` между ними. Метод `query` принимает условие текстом: имена столбцов пишутся как есть, условия соединяются словом `and`, значения-строки берутся в кавычки, отличные от кавычек всей строки. Результат тот же, что у маски в квадратных скобках: таблица подходящих строк.

In [ ]:
severe_day = dtp.query('dead_count >= 3 and light == "Светлое время суток"')   # три и более погибших днём
severe_day[['datetime', 'district', 'category', 'dead_count']].head()

In [ ]:
len(severe_day)                                                          # число таких ДТП

Переменная из ноутбука подставляется в условие через `@`: порог остаётся в коде один раз и меняется в одном месте.

In [ ]:
limit = 10                                                               # порог по раненым
dtp.query('injured_count >= @limit')[['datetime', 'district', 'category', 'injured_count']]

Для текстовых столбцов есть строковые методы через `str`: `contains` проверяет, есть ли подстрока, `startswith` — начинается ли значение с подстроки; оба возвращают маску. Три ночных значения освещения содержат слово «темное», и одна маска заменяет `isin` со списком из трёх значений. Знак `~` перед маской даёт отрицание.

In [ ]:
night = dtp['light'].str.contains('темное')                              # маска: ночные ДТП
night.sum()

In [ ]:
in_bryansk = dtp['district'].str.startswith('Брянск')                    # Брянская область и Брянский район
in_bryansk.sum(), (~in_bryansk).sum()

Ловушка строкового поиска тихая: ошибок нет, просто ничего не найдено. В источнике слово «темное» написано без буквы ё, и поиск с ё даёт ноль совпадений. Значения копируются из вывода `value_counts`, а различие прописных и строчных букв снимает параметр `case=False`.

In [ ]:
dtp['light'].str.contains('тёмное').sum()                                # с буквой ё: ноль совпадений

## 3. Сортировка

`sort_values` сортирует таблицу по столбцу; `ascending=False` ставит наибольшие значения первыми. Столбцы для отчёта задаём один раз списком.

In [ ]:
fields = ['datetime', 'district', 'category', 'dead_count', 'injured_count']   # поля для отчёта
dtp.sort_values('dead_count', ascending=False)[fields].head()

Список столбцов задаёт порядок при равенстве: среди ДТП с одинаковым числом погибших первыми идут ДТП с большим числом раненых.

In [ ]:
dtp.sort_values(['dead_count', 'injured_count'], ascending=False)[fields].head()

`nlargest` и `nsmallest` дают первые строки по одному столбцу без сортировки всей таблицы. При равных значениях порядок не определён: ДТП с одиннадцатью ранеными в таблице три, `nlargest` берёт первые два по порядку таблицы. Когда равенства важны, задаётся второй столбец сортировки.

In [ ]:
dtp.nlargest(3, 'injured_count')[fields]                                 # три ДТП по раненым

Сортировка возвращает новую таблицу, исходная остаётся как была: чтобы работать с отсортированной, её надо присвоить переменной. Ячейка ниже сортирует и тут же показывает начало исходной таблицы: там прежние ДТП 31 и 133.

In [ ]:
dtp.sort_values('dead_count', ascending=False)                           # сортировка без присваивания
dtp.head(2)                                                              # таблица прежняя

In [ ]:
by_dead = dtp.sort_values('dead_count', ascending=False)                 # отсортированная копия
by_dead.head(2)[fields]

Индекс сортируется методом `sort_index`; он понадобится для сводок, где индексом стали годы или названия групп.

## 4. Новые столбцы

Присваивание по новому имени добавляет столбец: справа стоит выражение над другими столбцами, и оно считается для каждой строки сразу. Число пострадавших — сумма погибших и раненых.

In [ ]:
dtp['victims'] = dtp['dead_count'] + dtp['injured_count']                # пострадавшие: погибшие и раненые
dtp[['dead_count', 'injured_count', 'victims']].head(3)

Дата и время в столбце `datetime` пока строка вида `2015-01-01 08:20:00`; настоящей датой она станет на семинаре 6. Год достаётся срезом первых четырёх символов через `str`, и это тоже новый столбец, текстовый. `value_counts` перечисляет годы по убыванию числа ДТП, `sort_index` ставит их по порядку.

In [ ]:
dtp['year'] = dtp['datetime'].str[:4]                                    # год ДТП строкой
dtp['year'].value_counts().sort_index()                                  # ДТП по годам

С 2015 по 2025 год число ДТП с пострадавшими в таблице упало втрое: с 1 392 до 465. Это не только дороги: за десять лет менялись правила учёта и полнота выгрузки; о чём говорит такая динамика, вернёмся в разделе 7.

Когда новый столбец получается заменой значений по правилу, правило записывается словарём, а замену делает метод `map`: пять условий освещения превращаются в три значения времени суток. В словаре должны быть все значения столбца, иначе у несловарных строк будет пропуск.

In [ ]:
daytime = {'Светлое время суток': 'день',
           'Сумерки': 'сумерки',
           'В темное время суток, освещение включено': 'ночь',
           'В темное время суток, освещение отсутствует': 'ночь',
           'В темное время суток, освещение не включено': 'ночь'}         # правило замены
dtp['daytime'] = dtp['light'].map(daytime)                               # время суток из трёх значений
dtp['daytime'].value_counts()

Когда правило не перечисление, а условие, пишется обычная функция от одного значения, а метод `apply` применяет её к каждому значению столбца. Месяц стоит в строке даты с пятого по шестой символ, «01»…«12», и сравнивается как текст; квартал по месяцу — отчётный период, по которому ГИБДД сводит статистику.

In [ ]:
def quarter(month):
    if month in ('01', '02', '03'):
        return 'I'
    if month in ('04', '05', '06'):
        return 'II'
    if month in ('07', '08', '09'):
        return 'III'
    return 'IV'


dtp['quarter'] = dtp['datetime'].str[5:7].apply(quarter)                 # квартал по месяцу
dtp['quarter'].value_counts().sort_index()

Изменить часть значений столбца по условию — частая задача, и у неё есть ловушка. В столбце `district` у 3 615 ДТП стоит «Брянская область»: район не указан, и в сводках по районам это значение читается как весь регион. Подпишем такие строки «без района». Запись `dtp[маска]['district'] = …` меняет не таблицу, а временную копию строк: pandas печатает предупреждение `ChainedAssignmentError`, а столбец остаётся прежним; ячейка выполняется без ошибки.

In [ ]:
dtp[dtp['district'] == 'Брянская область']['district'] = 'без района'    # запись в копию: не работает
dtp['district'].value_counts().head(3)                                   # подписи прежние

Правильная запись — через `loc` с маской и именем столбца в одних скобках: pandas меняет саму таблицу.

In [ ]:
dtp.loc[dtp['district'] == 'Брянская область', 'district'] = 'без района'   # запись через loc
dtp['district'].value_counts().head(3)                                   # подпись заменена

## 5. Группировка

`groupby` делит таблицу на группы по значениям столбца; дальше указывается столбец и статистика, и pandas считает её в каждой группе, а результат собирает в Series с названиями групп в индексе. Сумма погибших по районам заменяет 29 масок.

In [ ]:
dtp.groupby('district')['dead_count'].sum().sort_values(ascending=False).head()   # погибшие по районам

Вторая строка — «без района»: 3 615 ДТП отнесены к области без указания района, и, судя по координатам, в основном это город Брянск, который отдельным значением не выделен. В отчёте такая строка требует подписи; в README данных это записано.

Группировать можно и логический столбец: `size` даёт число ДТП в группе, `mean` — долю строк со значением `True`. Список статистик передаётся методу `agg`. Доля ДТП с погибшими по видам ДТП, вместе с числом ДТП каждого вида:

In [ ]:
dtp['with_dead'] = dtp['dead_count'] > 0                                 # маска как столбец
dtp.groupby('category')['with_dead'].agg(['size', 'mean']).sort_values('mean', ascending=False).round(3).head()

Первые строки заняты редкими видами ДТП: доля на пяти или шести ДТП ничего не значит, и столбец `size` рядом с долей не даёт этого забыть. О массовых видах разговор в разделе 6.

`agg` с именованными столбцами собирает несколько статистик по разным столбцам в одну таблицу: имя нового столбца слева, пара «столбец, статистика» справа. Сводка по годам отвечает на первый вопрос занятия.

In [ ]:
by_year = dtp.groupby('year').agg(accidents=('dead_count', 'size'),
                                  dead=('dead_count', 'sum'),
                                  injured=('injured_count', 'sum'))     # сводка по годам
by_year

Группировать можно не только таблицу по столбцу, но и один Series по другому: маска ночных ДТП из раздела 2, сгруппированная по столбцу с годом, даёт долю ночных ДТП в каждом году. Группировка сразу по двум столбцам тоже возможна, список имён в `groupby`, но читать такой результат тяжело; для двух признаков есть сводная таблица в разделе 6.

In [ ]:
night.groupby(dtp['year']).mean().round(3)                               # доля ночных ДТП по годам

Статистика по всей группировке без указания столбца пытается посчитать её для каждого столбца таблицы, а среднее по тексту не считается: ошибка `TypeError` в следующей ячейке намеренная. Столбцы выбираются списком до статистики.

In [ ]:
dtp.groupby('district').mean()                                           # среднее по тексту: ошибка

In [ ]:
dtp.groupby('district')[['dead_count', 'injured_count']].mean().round(2).head(3)   # средние по числовым столбцам

## 6. Сводные таблицы

Группировка по двум столбцам читается тяжело: значения одной группировки уходят в строки, другой — в столбцы. Так устроена сводная таблица: `pivot_table` получает таблицу, столбец со значениями `values`, столбец для строк `index`, столбец для столбцов `columns` и статистику `aggfunc`; `margins=True` добавляет итоги по строкам и столбцам. Погибшие по годам и времени суток:

In [ ]:
pd.pivot_table(dtp, values='dead_count', index='year', columns='daytime',
               aggfunc='sum', margins=True)                              # погибшие: год и время суток

Ночью за одиннадцать лет погибло 783 человека, днём 767, при том что ночных ДТП почти вдвое меньше дневных: 3 485 против 6 297. Число ночных смертей падало быстрее: 144 в 2015 году против 39 в 2024-м.

Сочетания, которых в данных нет, в сводной таблице становятся пропусками: ни одного ДТП в снегопад в третьем квартале, и в ячейке стоит `NaN`, а не ноль.

In [ ]:
pd.pivot_table(dtp, values='dead_count', index='quarter', columns='weather',
               aggfunc='sum')                                            # погибшие: квартал и погода

`fill_value=0` подставляет ноль вместо пропуска, `margins=True` добавляет итоги: такая таблица готова для отчёта.

In [ ]:
pd.pivot_table(dtp, values='dead_count', index='quarter', columns='weather',
               aggfunc='sum', fill_value=0, margins=True)                # то же с нулями и итогами

Когда считаются не суммы, а число строк по двум категориям, короче `crosstab`: два столбца, в ячейках число ДТП. С `normalize='index'` каждая строка делится на свою сумму, и получаются доли внутри вида ДТП. Третий вопрос занятия, о тяжести по видам, для трёх массовых видов:

In [ ]:
kinds = ['Столкновение', 'Наезд на пешехода', 'Съезд с дороги']           # три массовых вида ДТП
pd.crosstab(dtp['category'], dtp['severity']).loc[kinds]                 # ДТП по виду и тяжести

In [ ]:
pd.crosstab(dtp['category'], dtp['severity'], normalize='index').round(3).loc[kinds]   # доли внутри вида

Столкновений больше всего, но доля ДТП с погибшими у них ниже: 0.118 против 0.165 у наездов на пешеходов и 0.178 у съездов с дороги. Правило выбора: `crosstab` считает строки по двум категориям, `pivot_table` считает статистику столбца значений.

## 7. Мини-анализ и итоги

Первый вопрос, динамика по годам: к сводке `by_year` добавляется доля ДТП с погибшими. Столбец сводки создаётся присваиванием, как столбец таблицы, а справа стоит результат группировки по тем же годам: pandas подставляет значения по совпадающим меткам индекса.

In [ ]:
by_year['fatal_share'] = dtp.groupby('year')['with_dead'].mean().round(3)   # доля ДТП с погибшими
by_year

Число ДТП упало втрое, число погибших вдвое, а доля ДТП с погибшими выросла с 0.137 до 0.215. Такое сочетание может означать, что из учёта выпадают лёгкие ДТП, а не что дороги стали опаснее. Предположение проверяется тем же `crosstab`: число ДТП по годам и тяжести.

In [ ]:
pd.crosstab(dtp['year'], dtp['severity'])[['Легкий', 'Тяжёлый', 'С погибшими']]   # ДТП по годам и тяжести

Лёгких ДТП с 576 в 2015 году до 46 в 2025-м, а в 2020 году ни одного; тяжёлых и с погибшими за то же время стало вдвое меньше. Доля ДТП с погибшими росла потому, что из таблицы почти исчезли лёгкие ДТП: это свойство учёта, и сравнивать годы по этой доле нельзя. Первое, что пишется в отчёт.

Второй вопрос, районы: сводка с числом ДТП и долей, из которой маской оставлены районы не меньше чем со 100 ДТП. Порог отсекает малые группы, где доля складывается из единиц; маска здесь применяется к сводке, а не к исходной таблице.

In [ ]:
by_district = dtp.groupby('district').agg(accidents=('with_dead', 'size'),
                                          fatal_share=('with_dead', 'mean'))   # ДТП и доля с погибшими
large = by_district[by_district['accidents'] >= 100]                     # районы от 100 ДТП
large.sort_values('fatal_share', ascending=False).round(3).head()

Мглинский, Климовский, Унеченский районы: каждое четвёртое ДТП с погибшими против 0.134 по области. Третий вопрос закрыт в разделе 6.

Малые группы обманывают и тогда, когда порог не поставлен. По погоде доля ДТП с погибшими в тумане 0.35 против 0.13 в ясную погоду, но это 60 ДТП за одиннадцать лет: 21 ДТП с погибшими. Случайно ли такое различие, научат проверять семинары 12–13; до тех пор рядом с долей всегда стоит число ДТП.

In [ ]:
dtp.groupby('weather')['with_dead'].agg(['size', 'mean']).round(3)       # по погоде: ДТП и доля

Сводка уходит в отчёт: `rename` со словарём даёт столбцам русские подписи, `to_csv` записывает таблицу в файл рядом с ноутбуком, названия районов из индекса становятся первым столбцом.

In [ ]:
report = large.sort_values('fatal_share', ascending=False).round(3)
report = report.rename(columns={'accidents': 'ДТП', 'fatal_share': 'доля с погибшими'})   # подписи для отчёта
report.to_csv('dtp-bryansk-districts.csv')                              # сводка в файл
report.head(3)

## Итоги

`query` записывает условие текстом, строковые методы `str.contains` и `str.startswith` дают маски по тексту; `sort_values` сортирует по одному или нескольким столбцам и возвращает новую таблицу, `nlargest` берёт первые строки по столбцу; новый столбец создаётся присваиванием, срезом строки, `map` со словарём или `apply` с функцией, а часть значений меняется через `loc` с маской; `groupby` со статистикой или `agg` считает по группам, в том числе один Series по другому, `pivot_table` раскладывает статистику по двум признакам, `crosstab` считает строки по двум категориям, `normalize='index'` даёт доли, `fill_value` заполняет пустые сочетания. Рядом с долей по группе всегда стоит число строк в группе, а вывод из одной сводки проверяется другой: доля ДТП с погибшими росла потому, что из учёта выпадали лёгкие ДТП. Переходите к упражнениям: откройте `exercises/seminar-05-tasks.ipynb`, формулировки в `exercises/seminar-05-tasks.md`; данные там по Московской области за 2022–2025 годы.